In [ ]:
"""
Moduł przeprowadzający testy wydajnościowe akceleracji 3D na układzie FPGA (PYNQ).

Skrypt weryfikuje i porównuje wydajność obliczeniową głównego procesora (CPU)
z układem logicznym FPGA (korzystającym z protokołu AXI DMA) w dwóch scenariuszach:
1. Test małej próby: wielokrotna iteracja obrotu kilku wierzchołków (badanie narzutu DMA).
2. Test dużej próby: jednorazowy transfer i obrót dużej chmury punktów (badanie wydajności strumieniowej).
"""

import numpy as np
import math
import time
from pynq import Overlay, allocate

print("[INIT] Konfiguracja sprzętu FPGA do testów wydajnościowych...")
overlay = Overlay("design_1_wrapper.xsa")
dma = overlay.axi_dma_0

def run_benchmark():
    """
    Uruchamia i mierzy czasy wykonania rotacji wierzchołków dla CPU oraz FPGA.

    Funkcja alokuje odpowiednie bufory w pamięci, inicjalizuje dane wejściowe
    i wykonuje rotacje na małym zbiorze danych w pętli oraz na dużym zbiorze
    danych w pojedynczym strumieniu, wyświetlając wyniki i współczynnik
    przyspieszenia (speedup).
    """
    print("\n" + "="*60)
    print("TEST 1: 1000 sekwencyjnych obrotów figury (8 wierzchołków)")
    print("Cel: Ukazanie wąskiego gardła komunikacji (AXI DMA Overhead)")
    print("="*60)

    V_SMALL = 8
    ITERATIONS = 1000
    angle_deg = 1.0

    tx_small = allocate(shape=(4 * V_SMALL,), dtype=np.int32)
    rx_small = allocate(shape=(3 * V_SMALL,), dtype=np.int32)

    for i in range(V_SMALL):
        tx_small[i*4:i*4+4] = [int(math.radians(angle_deg)*1024), 50, 50, 50]
    tx_small.flush()

    rad = math.radians(angle_deg)
    cos_a, sin_a = math.cos(rad), math.sin(rad)
    test_vertices = [(50, 50, 50) for _ in range(V_SMALL)]

    t_start_cpu = time.perf_counter()
    for _ in range(ITERATIONS):
        for j in range(V_SMALL):
            x, y, z = test_vertices[j]
            nx = x * cos_a - y * sin_a
            ny = x * sin_a + y * cos_a
            test_vertices[j] = (nx, ny, z)
    t_cpu_1 = (time.perf_counter() - t_start_cpu) * 1000

    t_start_fpga = time.perf_counter()
    for _ in range(ITERATIONS):
        dma.recvchannel.transfer(rx_small)
        dma.sendchannel.transfer(tx_small)
        dma.sendchannel.wait()
        dma.recvchannel.wait()
    t_fpga_1 = (time.perf_counter() - t_start_fpga) * 1000

    print(f" -> Czas CPU:  {t_cpu_1:8.2f} ms ")
    print(f" -> Czas FPGA: {t_fpga_1:8.2f} ms ")

    del tx_small
    del rx_small

    print("\n" + "="*60)
    print("TEST 2: Pojedynczy obrót gigantycznej chmury (200 000 wierzchołków)")
    print("Cel: Demonstracja potoku obliczeniowego CORDIC (100 MHz)")
    print("="*60)

    V_LARGE = 200000

    print(f"Alokacja w CMA dla {V_LARGE} wierzchołków...")
    tx_large = allocate(shape=(4 * V_LARGE,), dtype=np.int32)
    rx_large = allocate(shape=(3 * V_LARGE,), dtype=np.int32)

    angle_hw = int(math.radians(angle_deg)*1024)
    tx_large[0::4] = angle_hw
    tx_large[1::4] = 50
    tx_large[2::4] = 50
    tx_large[3::4] = 50
    tx_large.flush()

    test_large_x = np.full(V_LARGE, 50.0)
    test_large_y = np.full(V_LARGE, 50.0)

    print("Rozpoczęcie obliczeń na CPU...")
    t_start_cpu = time.perf_counter()
    for j in range(V_LARGE):
        nx = test_large_x[j] * cos_a - test_large_y[j] * sin_a
        ny = test_large_x[j] * sin_a + test_large_y[j] * cos_a
    t_cpu_2 = (time.perf_counter() - t_start_cpu) * 1000

    print("Rozpoczęcie transferu z potokiem FPGA...")
    t_start_fpga = time.perf_counter()
    dma.recvchannel.transfer(rx_large)
    dma.sendchannel.transfer(tx_large)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t_fpga_2 = (time.perf_counter() - t_start_fpga) * 1000

    print(f" -> Czas CPU:  {t_cpu_2:8.2f} ms")
    print(f" -> Czas FPGA: {t_fpga_2:8.2f} ms")

    if t_cpu_2 > t_fpga_2:
        speedup = t_cpu_2 / t_fpga_2
        print(f"\n[WYNIK] FPGA jest obiektywnie {speedup:.1f}x szybsze w trybie strumieniowym.")

    del tx_large
    del rx_large

try:
    run_benchmark()
except Exception as e:
    print(f"Błąd podczas testu: {e}")